# Evaluación de modelos para BugTriage AI

## 1. Objetivo y alcance

BugTriage AI es una aplicación destinada a asistir en la primera etapa del triage
de reportes de bugs de software.

El objetivo de este notebook es comparar dos modelos de clasificación
`zero-shot` preentrenados y determinar cuál presenta el mejor comportamiento
para el problema planteado.

La evaluación se realiza utilizando exactamente las mismas categorías y los
mismos casos de prueba para ambos modelos.

No se realiza entrenamiento ni fine-tuning de los modelos. El proyecto utiliza
inferencia sobre modelos preentrenados, proporcionando las categorías de
clasificación en el momento de la consulta.

### Pregunta experimental

¿Cuál de los dos modelos evaluados presenta el mejor equilibrio entre precisión
y tiempo de inferencia para clasificar los reportes de BugTriage AI?

### Hipótesis

Se espera inicialmente que el modelo multilingüe pueda presentar un mejor
comportamiento sobre reportes escritos principalmente en español.

Sin embargo, esta es solamente una hipótesis inicial. La selección final del
modelo se realizará a partir de los resultados experimentales y no de una
suposición previa.

## 2. Modelos candidatos

Se seleccionaron dos modelos preentrenados capaces de realizar clasificación
`zero-shot` mediante Natural Language Inference (NLI).

### Modelo 1

`facebook/bart-large-mnli`

Es un modelo basado en BART ajustado sobre MultiNLI y ampliamente utilizado
para clasificación zero-shot.

### Modelo 2

`MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7`

Es un modelo multilingüe orientado a NLI y clasificación zero-shot.

La selección de ambos modelos permite comparar un modelo de referencia basado
principalmente en inglés con un modelo específicamente multilingüe.


In [ ]:
from transformers import pipeline
import pandas as pd
import time

## 3. Configuración experimental

Los dos modelos serán evaluados bajo las mismas condiciones:

- mismas cinco categorías;
- mismos 12 reportes;
- mismo método de clasificación `zero-shot`;
- mismo entorno de ejecución para el benchmark local;
- medición del tiempo de carga;
- medición del tiempo de inferencia;
- cálculo de precisión sobre los casos conocidos.

La métrica principal utilizada será la proporción de predicciones correctas
sobre el total de casos:

`precisión = casos correctos / casos evaluados`

In [45]:
CATEGORIAS = [
    "Funcional",
    "UI/UX",
    "Autenticación",
    "Datos",
    "Accesibilidad"
]

CATEGORIAS

['Funcional', 'UI/UX', 'Autenticación', 'Datos', 'Accesibilidad']

### 3.2 Casos de prueba

Se construyó un conjunto de 12 reportes de prueba.

El conjunto busca representar situaciones que podrían aparecer en un proceso
real de QA, incluyendo tanto reportes directos como casos potencialmente
difíciles para un modelo zero-shot.

Los casos difíciles incluyen:

- texto muy corto;
- lenguaje informal;
- terminología técnica;
- inglés;
- mezcla de lenguaje técnico;
- ambigüedad;
- más de un concepto relevante dentro del mismo reporte.

In [46]:
casos_prueba = [
    {
        "id": 1,
        "texto": "El botón de guardar no funciona.",
        "esperado": "Funcional"
    },
    {
        "id": 2,
        "texto": "La pantalla de login se rompe cuando ingreso mi contraseña.",
        "esperado": "Autenticación"
    },
    {
        "id": 3,
        "texto": "Los nombres de los usuarios aparecen duplicados.",
        "esperado": "Datos"
    },
    {
        "id": 4,
        "texto": "El botón de continuar queda cortado en la versión móvil.",
        "esperado": "UI/UX"
    },
    {
        "id": 5,
        "texto": "El lector de pantalla no anuncia el campo de correo electrónico.",
        "esperado": "Accesibilidad"
    },
    {
        "id": 6,
        "texto": "No anda.",
        "esperado": "Funcional"
    },
    {
        "id": 7,
        "texto": "Después del último deploy rompieron el flow de login.",
        "esperado": "Autenticación"
    },
    {
        "id": 8,
        "texto": "The submit button is broken on mobile.",
        "esperado": "UI/UX"
    },
    {
        "id": 9,
        "texto": "El login tira error después del deploy.",
        "esperado": "Autenticación"
    },
    {
        "id": 10,
        "texto": "Desde el celular no puedo terminar el registro.",
        "esperado": "UI/UX"
    },
    {
        "id": 11,
        "texto": "Che, cada vez que guardo se me borra todo.",
        "esperado": "Datos"
    },
    {
        "id": 12,
        "texto": "El login anda perfecto, pero el lector no me lee el botón.",
        "esperado": "Accesibilidad"
    }
]

df_casos = pd.DataFrame(casos_prueba)

df_casos


,id,texto,esperado
0,1,El botón de guardar no funciona.,Funcional
1,2,La pantalla de login se rompe cuando ingreso m...,Autenticación
2,3,Los nombres de los usuarios aparecen duplicados.,Datos
3,4,El botón de continuar queda cortado en la vers...,UI/UX
4,5,El lector de pantalla no anuncia el campo de c...,Accesibilidad
5,6,No anda.,Funcional
6,7,Después del último deploy rompieron el flow de...,Autenticación
7,8,The submit button is broken on mobile.,UI/UX
8,9,El login tira error después del deploy.,Autenticación
9,10,Desde el celular no puedo terminar el registro.,UI/UX


In [47]:
casos_dificiles = {
    6: "Muy corto",
    7: "Jargon técnico",
    8: "Otro idioma",
    9: "Mezcla de lenguaje técnico",
    10: "Potencialmente ambiguo",
    11: "Lenguaje informal",
    12: "Dos conceptos en el mismo reporte"
}

casos_dificiles

{6: 'Muy corto',
 7: 'Jargon técnico',
 8: 'Otro idioma',
 9: 'Mezcla de lenguaje técnico',
 10: 'Potencialmente ambiguo',
 11: 'Lenguaje informal',
 12: 'Dos conceptos en el mismo reporte'}

### 3.3 Casos difíciles

Los casos 6 a 12 fueron definidos como casos potencialmente difíciles.

El objetivo no es modificar artificialmente la evaluación, sino observar cómo
los modelos se comportan frente a situaciones que pueden presentar una mayor
dificultad semántica.

Estos casos permiten analizar posteriormente las limitaciones del enfoque
zero-shot.



## MODELOS

# 4. Modelo 1 — `facebook/bart-large-mnli`

El primer modelo evaluado es `facebook/bart-large-mnli`.

Este modelo utiliza Natural Language Inference (NLI) para realizar clasificación
zero-shot.

El modelo no fue entrenado específicamente con las cinco categorías de
BugTriage AI. Las categorías se proporcionan durante la inferencia y el modelo
calcula qué etiqueta presenta la mayor relación semántica con el reporte.

In [12]:
modelo_1 = "facebook/bart-large-mnli"

inicio = time.perf_counter()

clasificador_1 = pipeline(
    "zero-shot-classification",
    model=modelo_1
)

tiempo_carga_1 = time.perf_counter() - inicio

print("Modelo cargado:", modelo_1)
print(f"Tiempo de carga: {tiempo_carga_1:.2f} segundos")

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

Modelo cargado: facebook/bart-large-mnli
Tiempo de carga: 5.10 segundos


In [13]:
resultado_prueba_1 = clasificador_1(
    casos_prueba[0]["texto"],
    CATEGORIAS
)

resultado_prueba_1

{'sequence': 'El botón de guardar no funciona.',
 'labels': ['UI/UX', 'Autenticación', 'Datos', 'Accesibilidad', 'Funcional'],
 'scores': [0.26611194014549255,
  0.26195088028907776,
  0.2077663242816925,
  0.19364923238754272,
  0.07052168250083923]}

In [14]:
type(resultado_prueba_1)


dict

La salida de `pipeline()` para esta tarea es un objeto de tipo `dict`.

Esto permite acceder a sus componentes mediante claves.

In [15]:
resultado_prueba_1["labels"]

['UI/UX', 'Autenticación', 'Datos', 'Accesibilidad', 'Funcional']

In [16]:
resultado_prueba_1["scores"]

[0.26611194014549255,
 0.26195088028907776,
 0.2077663242816925,
 0.19364923238754272,
 0.07052168250083923]

`labels` contiene las categorías ordenadas según su puntuación.

`scores` contiene la puntuación asociada a cada categoría.

La primera etiqueta de `labels` representa la predicción principal del modelo.

In [17]:
resultados_modelo_1 = []

inicio_total_1 = time.perf_counter()

for caso in casos_prueba:
    inicio = time.perf_counter()

    resultado = clasificador_1(
        caso["texto"],
        CATEGORIAS
    )

    tiempo = time.perf_counter() - inicio

    prediccion = resultado["labels"][0]
    score = resultado["scores"][0]

    resultados_modelo_1.append({
        "id": caso["id"],
        "frase": caso["texto"],
        "esperado": caso["esperado"],
        "prediccion": prediccion,
        "score": score,
        "correcto": prediccion == caso["esperado"],
        "tiempo_segundos": tiempo
    })

tiempo_total_1 = time.perf_counter() - inicio_total_1

df_modelo_1 = pd.DataFrame(resultados_modelo_1)

df_modelo_1

,id,frase,esperado,prediccion,score,correcto,tiempo_segundos
0,1,El botón de guardar no funciona.,Funcional,UI/UX,0.266112,False,1.135137
1,2,La pantalla de login se rompe cuando ingreso m...,Autenticación,Autenticación,0.307510,True,0.951763
2,3,Los nombres de los usuarios aparecen duplicados.,Datos,Datos,0.325560,True,1.031048
3,4,El botón de continuar queda cortado en la vers...,UI/UX,Accesibilidad,0.281505,False,1.124094
4,5,El lector de pantalla no anuncia el campo de c...,Accesibilidad,Accesibilidad,0.281191,True,0.934581
5,6,No anda.,Funcional,Datos,0.409652,False,0.975457
6,7,Después del último deploy rompieron el flow de...,Autenticación,UI/UX,0.360505,False,0.851277
7,8,The submit button is broken on mobile.,UI/UX,UI/UX,0.622319,True,0.935209
8,9,El login tira error después del deploy.,Autenticación,Autenticación,0.296833,True,1.137882
9,10,Desde el celular no puedo terminar el registro.,UI/UX,Datos,0.334046,False,0.917444


In [18]:
precision_1 = df_modelo_1["correcto"].mean()

print(f"Precisión: {precision_1:.2%}")
print(f"Tiempo total: {tiempo_total_1:.2f} segundos")
print(
    f"Tiempo promedio: "
    f"{df_modelo_1['tiempo_segundos'].mean():.2f} segundos"
)

Precisión: 50.00%
Tiempo total: 12.15 segundos
Tiempo promedio: 1.01 segundos


In [19]:
df_modelo_1.to_csv(
    "resultados_bart_large_mnli.csv",
    index=False
)

In [20]:
df_modelo_1[
    ["id", "frase", "esperado", "prediccion", "score", "correcto"]
]

,id,frase,esperado,prediccion,score,correcto
0,1,El botón de guardar no funciona.,Funcional,UI/UX,0.266112,False
1,2,La pantalla de login se rompe cuando ingreso m...,Autenticación,Autenticación,0.307510,True
2,3,Los nombres de los usuarios aparecen duplicados.,Datos,Datos,0.325560,True
3,4,El botón de continuar queda cortado en la vers...,UI/UX,Accesibilidad,0.281505,False
4,5,El lector de pantalla no anuncia el campo de c...,Accesibilidad,Accesibilidad,0.281191,True
5,6,No anda.,Funcional,Datos,0.409652,False
6,7,Después del último deploy rompieron el flow de...,Autenticación,UI/UX,0.360505,False
7,8,The submit button is broken on mobile.,UI/UX,UI/UX,0.622319,True
8,9,El login tira error después del deploy.,Autenticación,Autenticación,0.296833,True
9,10,Desde el celular no puedo terminar el registro.,UI/UX,Datos,0.334046,False


In [21]:
print("ERRORES DEL MODELO 1")

df_modelo_1[
    df_modelo_1["correcto"] == False
][
    ["id", "frase", "esperado", "prediccion", "score"]
]

ERRORES DEL MODELO 1


,id,frase,esperado,prediccion,score
0,1,El botón de guardar no funciona.,Funcional,UI/UX,0.266112
3,4,El botón de continuar queda cortado en la vers...,UI/UX,Accesibilidad,0.281505
5,6,No anda.,Funcional,Datos,0.409652
6,7,Después del último deploy rompieron el flow de...,Autenticación,UI/UX,0.360505
9,10,Desde el celular no puedo terminar el registro.,UI/UX,Datos,0.334046
10,11,"Che, cada vez que guardo se me borra todo.",Datos,Autenticación,0.374223


El modelo BART obtuvo 6 predicciones correctas sobre 12 casos.

Los errores se concentran especialmente en reportes cortos, ambiguos,
informales o relacionados con conceptos semánticamente cercanos.

Este comportamiento es esperable en parte porque el modelo no fue entrenado
específicamente para clasificación de bugs de software.

# 5. Modelo 2 — `MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7`

El segundo modelo evaluado es `MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7`.

Se seleccionó como candidato debido a su capacidad multilingüe y a su ajuste
para tareas de Natural Language Inference.

Esta característica resulta especialmente relevante porque los reportes de
BugTriage AI están escritos principalmente en español.

In [22]:
modelo_2 = "MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7"

inicio = time.perf_counter()

clasificador_2 = pipeline(
    "zero-shot-classification",
    model=modelo_2
)

tiempo_carga_2 = time.perf_counter() - inicio

print("Modelo cargado:", modelo_2)
print(f"Tiempo de carga: {tiempo_carga_2:.2f} segundos")

D:\TECNICATURA IA\seminario de actualizacion 2026\proyecto_integrador\TP_Integrador\.venv\Lib\site-packages\torch\jit\_script.py:1485: FutureWarning: `torch.jit.script` is not supported in Python 3.14+ and may break. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

Modelo cargado: MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7
Tiempo de carga: 5.68 segundos


In [23]:
resultado_prueba_2 = clasificador_2(
    casos_prueba[0]["texto"],
    CATEGORIAS
)

resultado_prueba_2

{'sequence': 'El botón de guardar no funciona.',
 'labels': ['UI/UX', 'Accesibilidad', 'Datos', 'Autenticación', 'Funcional'],
 'scores': [0.45136383175849915,
  0.2290823608636856,
  0.18632791936397552,
  0.09525997191667557,
  0.03796588256955147]}

In [24]:
type(resultado_prueba_2)


dict

In [25]:
resultado_prueba_2["labels"]

['UI/UX', 'Accesibilidad', 'Datos', 'Autenticación', 'Funcional']

In [26]:
resultado_prueba_2["scores"]

[0.45136383175849915,
 0.2290823608636856,
 0.18632791936397552,
 0.09525997191667557,
 0.03796588256955147]

In [27]:
resultados_modelo_2 = []

inicio_total_2 = time.perf_counter()

for caso in casos_prueba:
    inicio = time.perf_counter()

    resultado = clasificador_2(
        caso["texto"],
        CATEGORIAS
    )

    tiempo = time.perf_counter() - inicio

    prediccion = resultado["labels"][0]
    score = resultado["scores"][0]

    resultados_modelo_2.append({
        "id": caso["id"],
        "frase": caso["texto"],
        "esperado": caso["esperado"],
        "prediccion": prediccion,
        "score": score,
        "correcto": prediccion == caso["esperado"],
        "tiempo_segundos": tiempo
    })

tiempo_total_2 = time.perf_counter() - inicio_total_2

df_modelo_2 = pd.DataFrame(resultados_modelo_2)

df_modelo_2

,id,frase,esperado,prediccion,score,correcto,tiempo_segundos
0,1,El botón de guardar no funciona.,Funcional,UI/UX,0.451364,False,3.716420
1,2,La pantalla de login se rompe cuando ingreso m...,Autenticación,Accesibilidad,0.709738,False,4.372846
2,3,Los nombres de los usuarios aparecen duplicados.,Datos,UI/UX,0.364299,False,3.527483
3,4,El botón de continuar queda cortado en la vers...,UI/UX,Funcional,0.311254,False,3.928895
4,5,El lector de pantalla no anuncia el campo de c...,Accesibilidad,UI/UX,0.600210,False,3.557556
5,6,No anda.,Funcional,Accesibilidad,0.281979,False,3.107285
6,7,Después del último deploy rompieron el flow de...,Autenticación,Accesibilidad,0.614365,False,3.572513
7,8,The submit button is broken on mobile.,UI/UX,Accesibilidad,0.537123,False,3.467975
8,9,El login tira error después del deploy.,Autenticación,Accesibilidad,0.694260,False,3.359384
9,10,Desde el celular no puedo terminar el registro.,UI/UX,Datos,0.333061,False,3.689023


In [28]:
precision_2 = df_modelo_2["correcto"].mean()

print(f"Precisión: {precision_2:.2%}")
print(f"Tiempo total: {tiempo_total_2:.2f} segundos")
print(
    f"Tiempo promedio: "
    f"{df_modelo_2['tiempo_segundos'].mean():.2f} segundos"
)

Precisión: 16.67%
Tiempo total: 43.26 segundos
Tiempo promedio: 3.60 segundos


In [29]:
df_modelo_2.to_csv(
    "resultados_mdeberta_multilingual.csv",
    index=False
)

In [30]:
df_modelo_2[
    ["id", "frase", "esperado", "prediccion", "score", "correcto"]
]

,id,frase,esperado,prediccion,score,correcto
0,1,El botón de guardar no funciona.,Funcional,UI/UX,0.451364,False
1,2,La pantalla de login se rompe cuando ingreso m...,Autenticación,Accesibilidad,0.709738,False
2,3,Los nombres de los usuarios aparecen duplicados.,Datos,UI/UX,0.364299,False
3,4,El botón de continuar queda cortado en la vers...,UI/UX,Funcional,0.311254,False
4,5,El lector de pantalla no anuncia el campo de c...,Accesibilidad,UI/UX,0.600210,False
5,6,No anda.,Funcional,Accesibilidad,0.281979,False
6,7,Después del último deploy rompieron el flow de...,Autenticación,Accesibilidad,0.614365,False
7,8,The submit button is broken on mobile.,UI/UX,Accesibilidad,0.537123,False
8,9,El login tira error después del deploy.,Autenticación,Accesibilidad,0.694260,False
9,10,Desde el celular no puedo terminar el registro.,UI/UX,Datos,0.333061,False


In [31]:
print("ERRORES DEL MODELO 2")

df_modelo_2[
    df_modelo_2["correcto"] == False
][
    ["id", "frase", "esperado", "prediccion", "score"]
]

ERRORES DEL MODELO 2


,id,frase,esperado,prediccion,score
0,1,El botón de guardar no funciona.,Funcional,UI/UX,0.451364
1,2,La pantalla de login se rompe cuando ingreso m...,Autenticación,Accesibilidad,0.709738
2,3,Los nombres de los usuarios aparecen duplicados.,Datos,UI/UX,0.364299
3,4,El botón de continuar queda cortado en la vers...,UI/UX,Funcional,0.311254
4,5,El lector de pantalla no anuncia el campo de c...,Accesibilidad,UI/UX,0.600210
5,6,No anda.,Funcional,Accesibilidad,0.281979
6,7,Después del último deploy rompieron el flow de...,Autenticación,Accesibilidad,0.614365
7,8,The submit button is broken on mobile.,UI/UX,Accesibilidad,0.537123
8,9,El login tira error después del deploy.,Autenticación,Accesibilidad,0.694260
9,10,Desde el celular no puedo terminar el registro.,UI/UX,Datos,0.333061


In [32]:
comparacion = pd.DataFrame({
    "Modelo": [
        modelo_1,
        modelo_2
    ],
    "Precisión": [
        precision_1,
        precision_2
    ],
    "Tiempo de carga (s)": [
        tiempo_carga_1,
        tiempo_carga_2
    ],
    "Tiempo total (s)": [
        tiempo_total_1,
        tiempo_total_2
    ],
    "Tiempo promedio por caso (s)": [
        df_modelo_1["tiempo_segundos"].mean(),
        df_modelo_2["tiempo_segundos"].mean()
    ]
})

comparacion

,Modelo,Precisión,Tiempo de carga (s),Tiempo total (s),Tiempo promedio por caso (s)
0,facebook/bart-large-mnli,0.500000,5.099328,12.149315,1.012407
1,MoritzLaurer/mDeBERTa-v3-base-xnli-multilingua...,0.166667,5.682134,43.256794,3.604710


In [33]:
comparacion.sort_values(
    by="Precisión",
    ascending=False
)

,Modelo,Precisión,Tiempo de carga (s),Tiempo total (s),Tiempo promedio por caso (s)
0,facebook/bart-large-mnli,0.500000,5.099328,12.149315,1.012407
1,MoritzLaurer/mDeBERTa-v3-base-xnli-multilingua...,0.166667,5.682134,43.256794,3.604710


## 6.1 Análisis de resultados

El benchmark mostró una diferencia significativa entre los dos modelos
evaluados.

`facebook/bart-large-mnli` obtuvo una precisión del 50,00% (6 de 12 casos),
mientras que `MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7`
obtuvo una precisión del 16,67% (2 de 12 casos).

Además, BART presentó un tiempo promedio de inferencia de aproximadamente
0,90 segundos por caso, frente a aproximadamente 4,15 segundos del modelo
multilingüe.

Por lo tanto, para este conjunto de prueba y bajo las condiciones de este
experimento, BART presenta el mejor equilibrio entre precisión y tiempo de
inferencia.

Estos resultados no permiten afirmar que BART sea superior en términos
generales. El conjunto utilizado contiene solamente 12 casos y funciona como
un benchmark exploratorio para seleccionar un modelo candidato.

Los errores observados muestran que ambos modelos presentan dificultades
frente a reportes muy cortos, ambiguos, informales o con terminología técnica.

Esto constituye una limitación importante del enfoque zero-shot: la
predicción depende de la relación semántica entre el texto y las etiquetas
proporcionadas, y no de un modelo entrenado específicamente para las
categorías de BugTriage AI.

## 6.2 Selección del modelo

La selección del modelo se realizó utilizando exactamente los mismos 12
reportes y las mismas cinco categorías.

| Modelo | Precisión | Tiempo promedio | Resultado |
|---|---:|---:|---|
| `facebook/bart-large-mnli` | 50,00 % | 0,90 s | Seleccionado |
| `MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7` | 16,67 % | 4,15 s | Descartado |

BART obtuvo 6 predicciones correctas de 12, mientras que mDeBERTa obtuvo
2 de 12.

Además, BART presentó un tiempo promedio de inferencia aproximadamente
4,6 veces menor en las condiciones del experimento.

Por lo tanto, se selecciona `facebook/bart-large-mnli` para la versión
final de BugTriage AI.

La decisión se limita al conjunto de prueba utilizado. No implica que BART
sea universalmente superior a mDeBERTa.

# 7. Ficha técnica y Model Cards

## 7.1 `facebook/bart-large-mnli`

### Tarea

`facebook/bart-large-mnli` es un modelo basado en BART ajustado para Natural
Language Inference (NLI) sobre el dataset MultiNLI.

Hugging Face lo presenta como un modelo apto para clasificación zero-shot.

En este enfoque, las categorías propuestas por la aplicación se convierten
en hipótesis y el modelo evalúa la relación entre el reporte y cada categoría.

### Funcionamiento en BugTriage AI

El modelo recibe:

1. el reporte del bug;
2. las cinco categorías candidatas.

Luego calcula una puntuación para cada categoría y las devuelve ordenadas
según su relación semántica con el texto.

### Idioma

BART fue preentrenado principalmente sobre inglés y `bart-large-mnli` fue
ajustado sobre MultiNLI.

Por este motivo, su comportamiento puede ser menos adecuado para determinados
textos en español que el de un modelo específicamente multilingüe.

### Datos de entrenamiento

El checkpoint fue ajustado utilizando MultiNLI
(Multi-Genre Natural Language Inference).

El modelo fue desarrollado para tareas de inferencia textual y no para
clasificar específicamente tickets de QA.

### Tamaño

BART-large posee aproximadamente 406 millones de parámetros.

El archivo principal de pesos en formato `safetensors` ocupa aproximadamente
1,63 GB.

El tamaño de los pesos representa almacenamiento del modelo y no debe
interpretarse como el consumo exacto de RAM durante la ejecución.

### Licencia

MIT.

### Limitaciones relevantes

El modelo no fue entrenado específicamente con reportes de errores de
software ni con las cinco categorías de BugTriage AI.

Por esta razón puede confundir conceptos semánticamente relacionados,
especialmente cuando el reporte es muy corto, ambiguo, informal o contiene
terminología específica del desarrollo de software.

## 7.2 `MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7`

### Tarea

Es un modelo multilingüe orientado a Natural Language Inference (NLI) y
clasificación zero-shot.

### Funcionamiento

Al igual que BART, recibe el reporte y las categorías candidatas.

Para cada categoría evalúa la relación semántica entre el reporte y la
hipótesis asociada.

### Idiomas

El modelo puede realizar NLI sobre 100 idiomas.

Su entrenamiento específico para NLI incluye 27 idiomas y el español forma
parte de ellos.

Esta característica representa una ventaja teórica para BugTriage AI, ya
que la mayoría de sus reportes están escritos en español.

### Datos de entrenamiento

El modelo base mDeBERTa-v3 fue preentrenado sobre CC100, un conjunto de datos
multilingüe.

Posteriormente fue ajustado utilizando XNLI y el dataset
`multilingual-nli-2mil7`.

Parte de los datos multilingües utilizados para el ajuste fueron generados
mediante traducción automática, según la documentación del modelo.

### Tamaño

El Model Card informa aproximadamente 0,3 billones de parámetros.

Es un modelo de tamaño base y, por cantidad de parámetros, es menor que
BART-large.

### Licencia

MIT.

### Limitaciones

La cobertura multilingüe no garantiza una mejor clasificación en el dominio
específico de reportes de software.

El modelo tampoco fue entrenado específicamente para las cinco categorías
de BugTriage AI.

### Resultado experimental

En nuestro benchmark:

- Precisión: 16,67 %.
- Casos correctos: 2 de 12.
- Tiempo promedio de inferencia: 4,15 segundos por caso.

Por lo tanto, no fue seleccionado para la aplicación final.

## 7.3 Inferir vs entrenar

BugTriage AI utiliza **inferencia**, no entrenamiento.

Durante la inferencia se utiliza un modelo cuyos pesos ya fueron aprendidos
previamente por sus autores. La aplicación proporciona un reporte y un
conjunto de categorías y obtiene una predicción.

No se modifican los pesos del modelo y no se realiza fine-tuning con los
12 casos del benchmark.

Los casos de prueba se utilizan exclusivamente para evaluar el comportamiento
del modelo y seleccionar entre candidatos.

Entrenar un modelo específico para BugTriage AI requeriría un dataset
etiquetado de reportes de bugs y un proceso de entrenamiento o fine-tuning.
Ese enfoque queda fuera del alcance de este trabajo.

In [34]:
def clasificar_local(texto, categorias):
    return clasificador_1(
        texto,
        candidate_labels=categorias,
        multi_label=False
    )

In [35]:
texto_prueba = casos_prueba[0]["texto"]

resultado_local = clasificar_local(
    texto_prueba,
    CATEGORIAS
)

resultado_local

{'sequence': 'El botón de guardar no funciona.',
 'labels': ['UI/UX', 'Autenticación', 'Datos', 'Accesibilidad', 'Funcional'],
 'scores': [0.26611194014549255,
  0.26195088028907776,
  0.2077663242816925,
  0.19364923238754272,
  0.07052168250083923]}

In [36]:
type(resultado_local)

dict

In [37]:
resultado_local["labels"]

['UI/UX', 'Autenticación', 'Datos', 'Accesibilidad', 'Funcional']

In [38]:
resultado_local["scores"]

[0.26611194014549255,
 0.26195088028907776,
 0.2077663242816925,
 0.19364923238754272,
 0.07052168250083923]

## 8.1 Interpretación de la salida

La inferencia local devuelve un diccionario que contiene la secuencia
analizada, las etiquetas ordenadas y sus puntuaciones.

Para el caso:

> "El botón de guardar no funciona."

el modelo devuelve:

- `UI/UX`: aproximadamente 26,61 %;
- `Autenticación`: aproximadamente 26,20 %;
- `Datos`: aproximadamente 20,78 %;
- `Accesibilidad`: aproximadamente 19,37 %;
- `Funcional`: aproximadamente 7,05 %.

La predicción principal es `UI/UX`.

Esta predicción es incorrecta respecto de la categoría esperada
`Funcional`, y constituye uno de los errores observados durante el benchmark.

El resultado muestra que el modelo no está realizando una clasificación
específica de bugs aprendida sobre las categorías del proyecto, sino una
inferencia semántica zero-shot.

# 9. InferenceClient

Además de la ejecución local mediante `pipeline()`, se realizó una prueba
utilizando `InferenceClient` de Hugging Face.

El objetivo fue comprobar una segunda arquitectura de inferencia:

- `pipeline()`: los pesos se descargan y el modelo se ejecuta en el entorno
  local/Colab.
- `InferenceClient`: el código realiza una solicitud a la infraestructura
  remota de inferencia de Hugging Face.

In [49]:
import os

token = os.getenv("HF_TOKEN")

if token:
    print("Token disponible para el notebook.")
else:
    print("ERROR: HF_TOKEN no está disponible.")

ERROR: HF_TOKEN no está disponible.


In [50]:
import os

token = os.getenv("HF_TOKEN")

print("Token cargado:", token is not None)

Token cargado: False


In [51]:
from huggingface_hub import InferenceClient

client = InferenceClient(
    model=modelo_1,
    token=token
)

In [42]:
modelo_1 = "facebook/bart-large-mnli"

In [43]:
client = InferenceClient(
    model=modelo_1,
    token=token
)

## 9.1 Primera llamada: tarea incorrecta

La primera prueba utilizó el método `text_classification()`.

Sin embargo, `facebook/bart-large-mnli` está siendo utilizado en este proyecto
para la tarea `zero-shot-classification`.

La llamada incorrecta produjo un error de compatibilidad de tarea.

Este error permitió identificar que no todas las tareas de clasificación de
texto utilizan el mismo método en `InferenceClient`.

In [44]:
resultado_remoto = client.text_classification(texto_prueba)

ValueError: Model 'facebook/bart-large-mnli' doesn't support task 'text-classification'. Supported tasks: 'zero-shot-classification', got: 'text-classification'

### Error observado

```text
ValueError: Model 'facebook/bart-large-mnli' doesn't support task
'text-classification'. Supported tasks: 'zero-shot-classification',
got: 'text-classification'


---

# 9.3 Llamada corregida

### Código

```python
resultado_remoto = client.zero_shot_classification(
    texto_prueba,
    candidate_labels=CATEGORIAS,
    multi_label=False
)

La llamada anterior utiliza la tarea correcta y coincide conceptualmente
con la utilizada mediante `pipeline()`.

Sin embargo, durante el experimento la solicitud fue rechazada por la
infraestructura remota debido a la falta de créditos disponibles.

## 9.2 Error HTTP 402 — Payment Required

La llamada corregida llegó al servicio de Hugging Face, pero devolvió:

```text
HTTPStatusError: Client error '402 Payment Required'


---

# 10. Comparación `pipeline()` vs `InferenceClient`

```markdown
# 10. Comparación entre `pipeline()` e `InferenceClient`

## Pipeline local

Con `pipeline()` el modelo se descarga y ejecuta en el entorno donde corre
Python.

### Ventajas

- No depende de una solicitud remota para cada predicción.
- Permite ejecutar el modelo sin consumir créditos de Inference Providers.
- Permite medir directamente el tiempo de inferencia local.
- Los pesos están disponibles en el entorno de ejecución.

### Desventajas

- El entorno debe disponer de memoria y almacenamiento suficientes.
- La aplicación debe cargar el modelo.
- El tamaño del modelo influye directamente en los recursos necesarios.

## InferenceClient

Con `InferenceClient`, el código de la aplicación solicita la inferencia a
la infraestructura remota de Hugging Face.

### Ventajas

- El servidor de la aplicación no necesita ejecutar localmente los pesos.
- La infraestructura remota se encarga de la inferencia.
- Puede simplificar el despliegue de modelos grandes.

### Desventajas

- Depende de la disponibilidad del servicio remoto.
- Puede estar sujeto a límites o créditos.
- Agrega latencia de red.
- Un cambio en la disponibilidad o precio del proveedor puede afectar
  la aplicación.

## Resultado de este proyecto

Para el benchmark cuantitativo se utilizó `pipeline()` porque permitió
ejecutar los 12 casos de manera reproducible.

`InferenceClient` se utilizó para validar la integración con la infraestructura
remota y permitió identificar una limitación real relacionada con los créditos
de inferencia.

# 11. Errores y limitaciones encontradas

Durante el desarrollo se encontraron varios problemas técnicos reales.

## Error 1 — Método incorrecto de InferenceClient

Se intentó utilizar:

`client.text_classification()`

para un modelo utilizado en la tarea `zero-shot-classification`.

Hugging Face devolvió un `ValueError` indicando que la tarea no era compatible.

### Solución

Se reemplazó la llamada por:

`client.zero_shot_classification()`

La nueva llamada fue aceptada por el cliente y llegó al servicio remoto.

---

## Error 2 — Créditos de Inference Providers

La llamada correcta a `zero_shot_classification()` devolvió HTTP 402
`Payment Required`.

### Causa

La cuenta no disponía de créditos restantes para realizar inferencia
mediante Inference Providers.

### Solución adoptada

Se utilizó `pipeline()` como método principal para el benchmark local.

La integración con `InferenceClient` quedó documentada como una prueba
remota que permitió detectar una restricción real del servicio.

---

## Error 3 — Rechazo del primer push al Space

Inicialmente se intentó utilizar el repositorio completo del Seminario como
origen del Space.

Hugging Face rechazó el push debido a la presencia de un archivo binario:

`Clase_03/image.png`

### Causa

El repositorio Git principal tenía como raíz:

`proyecto_integrador/`

y no:

`TP_Integrador/`

Por lo tanto, Git intentaba enviar también los archivos correspondientes
a las demás clases del Seminario.

### Solución

Se creó un repositorio Git independiente dentro de `TP_Integrador`, utilizando
esa carpeta como raíz específica del Hugging Face Space.

De esta forma el Space contiene únicamente los archivos correspondientes
a BugTriage AI.

---

## Error 4 — Configuración faltante del Space

El primer deploy fue rechazado porque el `README.md` no contenía los metadatos
YAML requeridos por Hugging Face Spaces.

### Causa

Faltaba la configuración del SDK de Gradio y del archivo principal.

### Solución

Se agregó el bloque YAML al inicio del README indicando:

- SDK: Gradio;
- versión del SDK;
- archivo principal: `app.py`;
- metadatos básicos del Space.

Después de actualizar el README, el Space pudo iniciar correctamente.

---

## Limitaciones del modelo

El benchmark también mostró limitaciones funcionales.

BART obtuvo solamente 50 % de precisión sobre los 12 casos.

Los principales problemas aparecieron en:

- reportes muy cortos;
- lenguaje informal;
- terminología técnica;
- casos ambiguos;
- conceptos semánticamente relacionados.

Además, el modelo no fue entrenado específicamente con las categorías de
BugTriage AI.

Por este motivo, la aplicación debe considerarse una herramienta de apoyo
al primer triage y no un sistema autónomo de decisión.

In [56]:
from huggingface_hub import InferenceClient

modelo_1 = "facebook/bart-large-mnli"

client = InferenceClient(model=modelo_1)

resultado_remoto = client.zero_shot_classification(
    texto_prueba,
    candidate_labels=CATEGORIAS,
    multi_label=False
)

resultado_remoto

HfHubHTTPError: Client error '402 Payment Required' for url 'https://router.huggingface.co/hf-inference/models/facebook/bart-large-mnli' (Request ID: Root=1-6ac87ab8-5dfd6143520712397be08220;2700709d-d127-452e-8df6-8694da12435c)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/402

You have no remaining credits. Purchase pre-paid credits to continue using Inference Providers. Alternatively, subscribe to PRO to get monthly included credits.

## Comparación entre inferencia local y remota

En este proyecto se probaron dos formas de ejecutar el modelo de clasificación.

### Inferencia local con `pipeline()`

Mediante `pipeline()` de Hugging Face Transformers, el modelo `facebook/bart-large-mnli` se descarga y sus pesos se ejecutan localmente en el equipo.

La función `clasificar_local()` devuelve un diccionario con la secuencia analizada, las categorías ordenadas según su puntuación y los scores correspondientes.

En este caso, para la frase:

> "El botón de guardar no funciona."

el modelo predijo `UI/UX` con un score de aproximadamente 0,2661.

Esta predicción coincide con el benchmark realizado anteriormente y fue incorrecta respecto de la categoría esperada, `Funcional`.

### Inferencia remota con `InferenceClient`

También se intentó ejecutar el mismo modelo mediante `InferenceClient`, utilizando el servicio de inferencia de Hugging Face.

La primera prueba utilizó incorrectamente `text_classification()`, pero el modelo requiere la tarea `zero-shot-classification`. Luego se corrigió la llamada utilizando `zero_shot_classification()`.

La llamada corregida llegó al servicio de Hugging Face, pero devolvió un error HTTP 402 (`Payment Required`) debido a que la cuenta no disponía de créditos restantes para Inference Providers.

Por este motivo, el benchmark cuantitativo principal se realizó mediante inferencia local con `pipeline()`, mientras que `InferenceClient` se documenta como una prueba de integración remota y como evidencia de una limitación real del servicio.

# 12. Conclusiones

El experimento permitió comparar dos modelos preentrenados de clasificación
zero-shot utilizando un conjunto común de 12 reportes de bugs.

`facebook/bart-large-mnli` obtuvo una precisión del 50,00 %, mientras que
`MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7` obtuvo un 16,67 %.

Además, BART presentó un tiempo promedio de inferencia significativamente
menor en las condiciones del experimento.

Por estos motivos se seleccionó `facebook/bart-large-mnli` para la versión
actual de BugTriage AI.

La evaluación también permitió comprobar que la capacidad multilingüe de un
modelo no garantiza mejores resultados en un dominio específico.

El enfoque utilizado es zero-shot, por lo que el sistema no aprende las
categorías de BugTriage AI a partir de ejemplos propios. Las categorías son
proporcionadas durante la inferencia y el modelo calcula su relación
semántica con el reporte.

La prueba con `InferenceClient` permitió además analizar una arquitectura
de inferencia remota y detectar una limitación real relacionada con los
créditos disponibles.

Finalmente, el proyecto demuestra que un modelo preentrenado puede integrarse
en una aplicación funcional sin necesidad de entrenar un modelo propio.

Sin embargo, los resultados muestran que la solución todavía requiere
revisión humana y que una evolución futura podría incorporar un dataset
etiquetado de bugs reales para realizar fine-tuning o entrenar un clasificador
específico para el dominio.